<a href="https://colab.research.google.com/github/mcfvdesigns/Site-Turnover-Defects/blob/main/M4%7CU3_Assignment_train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import os, hashlib, zipfile, urllib.request, yaml
from pathlib import Path

DATA_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/Site.Turnover.Defects.v4i.yolov11.1.zip"
SHA256 = "443eced074cc61c29b5fd3e0b8a0ce3bb5d6de410312312acc5ce936a10a84bc"
ROOT = Path("/content/dataset")
ZIP_PATH = Path("/content/dataset.zip")

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def find_data_yaml(root: Path) -> Path:
    hits = list(root.rglob("data.yaml"))
    if not hits:
        raise FileNotFoundError(f"No data.yaml under {root}")
    return hits[0]

already_ready = (ROOT / "data.yaml").exists() or any(ROOT.glob("*/data.yaml"))
if not already_ready:
    ROOT.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)
    digest = sha256_file(ZIP_PATH)
    if digest != SHA256:
        ZIP_PATH.unlink(missing_ok=True)
        raise AssertionError(f"Checksum mismatch: {digest}")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(ROOT)

cfg_path = find_data_yaml(ROOT)
dataset_root = cfg_path.parent
cfg = yaml.safe_load(cfg_path.read_text())
cfg["path"] = str(dataset_root)
cfg["train"] = "train/images"
cfg["val"] = "valid/images"
if (dataset_root / "test" / "images").is_dir():
    cfg["test"] = "test/images"
else:
    cfg.pop("test", None)
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg)

{'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 5, 'names': ['cracked jamb', 'cracked tile', 'moisture stain', 'paint defect', 'uneven joint'], 'roboflow': {'workspace': 'skipper-_sky', 'project': 'site-turnover-defects', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/skipper-_sky/site-turnover-defects/dataset/4'}, 'path': '/content/dataset'}


In [20]:
!pip install -q ultralytics matplotlib opencv-python

import os
from pathlib import Path
import matplotlib.pyplot as plt
import cv2
from ultralytics import YOLO

In [23]:
import urllib.request
from pathlib import Path

# Direct download link to your best.pt release asset
WEIGHTS_URL = "https://github.com/mcfvdesigns/Site-Turnover-Defects/releases/download/v4.0/best.pt"
WEIGHTS_PATH = Path("best.pt")

if not WEIGHTS_PATH.exists():
    print("Downloading model weights...")
    try:
        # User-Agent header prevents GitHub HTTP 403 blocks
        req = urllib.request.Request(WEIGHTS_URL, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response, open(WEIGHTS_PATH, 'wb') as out_file:
            out_file.write(response.read())
        print("Weights downloaded successfully.")
    except Exception as e:
        print(f"Error downloading weights: {e}")

Weights downloaded successfully.


In [24]:
# Display the file structure of the downloaded dataset
!ls -R /content/dataset

/content/dataset:
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid

/content/dataset/test:
images	labels

/content/dataset/test/images:
39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.jpg
3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.jpg
3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.jpg
5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.jpg
6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.rf.e1e4e1cb8897a6f59a1ae0cb7e67ae04.jpg

/content/dataset/test/labels:
39C7A391-E9C7-4E96-9517-6D375401DEB8_heic.rf.5db3a079d7a90bf32e4eb67064162c4b.txt
3B0369CD-D1EE-472C-9F32-4B2AF3395297_heic.rf.a5cc760ef037a0c61012c6bef46a2a9a.txt
3C0BF6B7-BD58-4F89-9809-2541F0FF267A_heic.rf.b349f558e2e2c53882e350e3996622b9.txt
5D20FB4A-8995-4278-8FB0-38F0F16F3222_heic.rf.90e31dfec36c4df4657b053fab35876d.txt
6CCD60CE-BE5E-41A1-8675-DFD83020914E_heic.rf.e1e4e1cb8897a6f59a1ae0cb7e67